In [45]:
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from match.achilles.model import Achilles
from instant_match.components import (
    build_features
)

from instant_match.loader.startup import get_vectorizer_path

local_vectorizer_path = get_vectorizer_path("AR", "5")
project_id = "dh-global-sales-data-dev"

In [46]:
query = f"""
SELECT *
FROM `dh-global-sales-data-dev.achilles.training_candidates_new`
where country_iso = 'AR'
and model_version = '5'
and left_row_id not like 'ext_row_ar_%'
"""

df_training_candidates_new_ar_v5 = pd.read_gbq(query=query, project_id = project_id, progress_bar_type = 'tqdm')

/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████|


In [47]:
df_training_candidates_new_ar_v5.head()

,country_iso,left_row_id,left_name,left_street,left_lat,left_lng,left_phone_number,left_street_stop,left_name_stop,left_name_stop_phonetic,...,left_name_local_transliterated,right_name_local_transliterated,right_name_legal,model_id,col_type,left_registration_number,right_registration_number,data_category_type,left_area,right_area
0,AR,FP_AR----botsol---google---0016be34c780f2a9850...,heladería grido,av juan b cabral,-26.176845,-58.188091,+5493704356688,juan cabral,grido,KRT,...,,,None,AR,train,None,None,geo,None,None
1,AR,FP_AR----botsol---google---00783f6a11de455bae4...,caliú bariloche,mitre,-41.134573,-71.300615,+5492944102526,mitre,caliu bariloche,KL BRLX,...,,,None,AR,train,None,None,geo,None,None
2,AR,FP_AR----botsol---google---0116ff7418d3495bcbc...,farmacia asamblea,del progreso,-34.634623,-58.437247,+541149233529,progreso,farmacia asamblea,FRMX ASMBL,...,,,None,AR,train,None,None,geo,None,None
3,AR,FP_AR----botsol---google---0116ff7418d3495bcbc...,farmacia asamblea,del progreso,-34.634623,-58.437247,+541149233529,progreso,farmacia asamblea,FRMX ASMBL,...,,,None,AR,train,None,None,geo,None,None
4,AR,FP_AR----botsol---google---0118a190047f88b3a03...,despensa seba,aristobulo del valle,-32.880698,-68.820815,+542615267850,aristobulo valle,despensa seba,TSPNS SB,...,,,None,AR,test,None,None,geo,None,None


In [75]:
candidates = df_training_candidates_new_ar_v5.drop(columns=["model_id", "model_version", "col_type"])

In [76]:
features = [
    "country_iso",
    "left_row_id",
    "right_row_id",
    "levenshtein_name_stop_phonetic",
    "levenshtein_street_stop_phonetic",
    "wratio_name",
    "jaro_winkler_name",
    "jaro_winkler_name_stop",
    "jaro_winkler_name_local",
    "jaro_winkler_name_local_transliterated",
    "jaro_winkler_name_reversed",
    "jw_name_local_ascii_only",
    "jw_name_local_nonascii_only",
    "tokenset_name_local",
    "tokenset_name_legal",
    "tokenset_name_local_nonascii_only",
    "tokenset_name_local_ascii_only",
    "tokenset_name_local_transliterated",
    "tokenset_name",
    "tokenset_name_stop",
    "tokenset_street_stop",
    "wratio_name_stop",
    "wratio_street_stop",
    "cosine_name_stop",
    "haversine",
    "same_phone",
]

In [77]:
feature_pipeline = Achilles(
        model_id="AR",
        model_version="5",
        local_vectoriser_path=local_vectorizer_path,
        feature_list=features,
    )

/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator TfidfTransformer from version 0.24.2 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator TfidfVectorizer from version 0.24.2 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [78]:
features_df = build_features(candidates, feature_pipeline)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [79]:
features_df.head()

,country_iso,left_row_id,right_row_id,levenshtein_name_stop_phonetic,levenshtein_street_stop_phonetic,wratio_name,jaro_winkler_name,jaro_winkler_name_stop,jaro_winkler_name_local,jaro_winkler_name_local_transliterated,...,tokenset_name_local_ascii_only,tokenset_name_local_transliterated,tokenset_name,tokenset_name_stop,tokenset_street_stop,wratio_name_stop,wratio_street_stop,cosine_name_stop,haversine,same_phone
0,AR,FP_AR----botsol---google---0016be34c780f2a9850...,FP_AR----salesforce---salesforce---HK6H7Y,1.000000,0.142857,0.774074,0.672222,1.000000,0.672222,0.0,...,0.500000,0.0,0.814815,1.000000,0.173913,1.000000,0.173913,1.000000,0.015833,0
1,AR,FP_AR----botsol---google---00783f6a11de455bae4...,FP_AR----salesforce---salesforce---HK88J6,0.571429,0.000000,0.855000,0.609343,0.792593,0.609343,0.0,...,0.571429,0.0,0.750000,1.000000,0.153846,0.900000,0.225000,0.680860,0.291400,0
2,AR,FP_AR----botsol---google---0116ff7418d3495bcbc...,FP_AR----salesforce---salesforce---HLIHA4,0.100000,0.000000,0.684211,0.692020,0.439542,0.692020,0.0,...,0.666667,0.0,0.684211,0.413793,0.100000,0.393103,0.180000,0.079375,0.045309,0
3,AR,FP_AR----botsol---google---0116ff7418d3495bcbc...,FP_AR----salesforce---salesforce---4OV2F5,0.100000,0.000000,0.690909,0.597339,0.597339,0.597339,0.0,...,0.551724,0.0,0.727273,0.727273,0.125000,0.690909,0.125000,0.588214,0.177576,0
4,AR,FP_AR----botsol---google---0118a190047f88b3a03...,FP_AR----salesforce---salesforce---HZX7A5,0.777778,0.266667,0.855000,0.683955,0.926374,0.683955,0.0,...,0.625000,0.0,0.761905,0.814815,0.350000,0.814815,0.427500,0.674165,0.546689,0


In [80]:
def generate_vendor_document(name, name_local, street):
    return f"""name: {name}, name_local: {name_local}, street: {street}"""

In [81]:
def generate_sentence_transformer_embeddings(documents):
    batch_size = 128
    embeddings = []
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    for start_idx in tqdm(range(0, len(documents), batch_size)):
        end_idx = start_idx + batch_size
        batch = documents[start_idx:end_idx]
        embeddings_batch = model.encode(batch)
        embeddings.extend(embeddings_batch)
    return embeddings

In [82]:
def create_semantic_features(df):
    #Generate document
    print("Generate document")
    df["left_document"] = df.apply(lambda x: generate_vendor_document(x.left_name, x.left_name_local, x.left_street), axis=1)
    df["right_document"] = df.apply(lambda x: generate_vendor_document(x.right_name, x.right_name_local, x.right_street), axis=1)

    #Transform to list
    print("Transform to list")
    left_documents = df["left_document"].values.tolist()
    right_documents = df["right_document"].values.tolist()
    left_names = df["left_name"].values.tolist()
    right_names = df["right_name"].values.tolist()
    left_names_local = df["left_name_local"].values.tolist()
    right_names_local = df["right_name_local"].values.tolist()
    left_street = df["left_street"].values.tolist()
    right_street = df["right_street"].values.tolist()

    #Generate embeddings
    print("Generate embeddings")
    left_document_embeddings = generate_sentence_transformer_embeddings(left_documents)
    right_document_embeddings = generate_sentence_transformer_embeddings(right_documents)
    left_name_embeddings = generate_sentence_transformer_embeddings(left_names)
    right_name_embeddings = generate_sentence_transformer_embeddings(right_names)
    left_name_local_embeddings = generate_sentence_transformer_embeddings(left_names_local)
    right_name_local_embeddings = generate_sentence_transformer_embeddings(right_names_local)
    left_street_embeddings = generate_sentence_transformer_embeddings(left_street)
    right_street_embeddings = generate_sentence_transformer_embeddings(right_street)

    #Compute similarity
    print("Compute similarity")
    doc_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_document_embeddings, right_document_embeddings)]
    name_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_name_embeddings, right_name_embeddings)]
    name_local_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_name_local_embeddings, right_name_local_embeddings)]
    street_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_street_embeddings, right_street_embeddings)]

    #Store results
    print("Store results")
    df["doc_sem_similarity"] = doc_sem_similarity
    df["name_sem_similarity"] = name_sem_similarity
    df["name_local_sem_similarity"] = name_local_sem_similarity
    df["street_sem_similarity"] = street_sem_similarity
    
    return df

In [83]:
candidates_w_semantic_features = create_semantic_features(candidates)

Generate document
Transform to list
Generate embeddings


100%|███████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 12.13it/s]


Compute similarity
Store results


In [84]:
candidates_w_semantic_features.head()

,country_iso,left_row_id,left_name,left_street,left_lat,left_lng,left_phone_number,left_street_stop,left_name_stop,left_name_stop_phonetic,...,right_registration_number,data_category_type,left_area,right_area,left_document,right_document,doc_sem_similarity,name_sem_similarity,name_local_sem_similarity,street_sem_similarity
0,AR,FP_AR----botsol---google---0016be34c780f2a9850...,heladería grido,av juan b cabral,-26.176845,-58.188091,+5493704356688,juan cabral,grido,KRT,...,None,geo,None,None,"name: heladería grido, name_local: heladería g...","name: grido helado, name_local: grido helado, ...",0.777968,0.897337,0.897337,0.425440
1,AR,FP_AR----botsol---google---00783f6a11de455bae4...,caliú bariloche,mitre,-41.134573,-71.300615,+5492944102526,mitre,caliu bariloche,KL BRLX,...,None,geo,None,None,"name: caliú bariloche, name_local: caliú baril...","name: punto empanada bariloche, name_local: pu...",0.786771,0.775422,0.775422,0.600655
2,AR,FP_AR----botsol---google---0116ff7418d3495bcbc...,farmacia asamblea,del progreso,-34.634623,-58.437247,+541149233529,progreso,farmacia asamblea,FRMX ASMBL,...,None,geo,None,None,"name: farmacia asamblea, name_local: farmacia ...","name: supermercado asamblea, name_local: super...",0.802324,0.600213,0.600213,0.319795
3,AR,FP_AR----botsol---google---0116ff7418d3495bcbc...,farmacia asamblea,del progreso,-34.634623,-58.437247,+541149233529,progreso,farmacia asamblea,FRMX ASMBL,...,None,geo,None,None,"name: farmacia asamblea, name_local: farmacia ...","name: asamblea plaza, name_local: asamblea pla...",0.671722,0.301646,0.301646,0.495218
4,AR,FP_AR----botsol---google---0118a190047f88b3a03...,despensa seba,aristobulo del valle,-32.880698,-68.820815,+542615267850,aristobulo valle,despensa seba,TSPNS SB,...,None,geo,None,None,"name: despensa seba, name_local: despensa seba...","name: cerrado despensa santy, name_local: cerr...",0.845787,0.592930,0.592930,0.423565


In [85]:
features_df_w_semantic_sims = candidates_w_semantic_features.drop(columns=["haversine"]).merge(features_df, how="left", left_on=["country_iso", "left_row_id", "right_row_id"], right_on=["country_iso", "left_row_id", "right_row_id"])

In [74]:
features_df_w_semantic_sims.columns

Index(['country_iso', 'left_row_id', 'left_name', 'left_street', 'left_lat',
       'left_lng', 'left_phone_number', 'left_street_stop', 'left_name_stop',
       'left_name_stop_phonetic', 'left_street_stop_phonetic', 'right_row_id',
       'right_name', 'right_street', 'right_lat', 'right_lng',
       'right_phone_number', 'right_street_stop', 'right_name_stop',
       'right_name_stop_phonetic', 'right_street_stop_phonetic', 'label',
       'model_version', 'left_name_local', 'right_name_local',
       'left_name_local_transliterated', 'right_name_local_transliterated',
       'right_name_legal', 'left_registration_number',
       'right_registration_number', 'data_category_type', 'left_area',
       'right_area', 'left_document', 'right_document', 'doc_sem_similarity',
       'name_sem_similarity', 'name_local_sem_similarity',
       'street_sem_similarity', 'levenshtein_name_stop_phonetic',
       'levenshtein_street_stop_phonetic', 'wratio_name', 'jaro_winkler_name',
       'jaro_

In [87]:
['doc_sem_similarity',
       'name_sem_similarity', 'name_local_sem_similarity',
       'street_sem_similarity', 'levenshtein_name_stop_phonetic',
       'levenshtein_street_stop_phonetic', 'wratio_name', 'jaro_winkler_name',
       'jaro_winkler_name_stop', 'jaro_winkler_name_local',
       'jaro_winkler_name_local_transliterated', 'jaro_winkler_name_reversed',
       'jw_name_local_ascii_only', 'jw_name_local_nonascii_only',
       'tokenset_name_local', 'tokenset_name_legal',
       'tokenset_name_local_nonascii_only', 'tokenset_name_local_ascii_only',
       'tokenset_name_local_transliterated', 'tokenset_name',
       'tokenset_name_stop', 'tokenset_street_stop', 'wratio_name_stop',
       'wratio_street_stop', 'cosine_name_stop', 'haversine', 'same_phone']

['doc_sem_similarity',
 'name_sem_similarity',
 'name_local_sem_similarity',
 'street_sem_similarity',
 'levenshtein_name_stop_phonetic',
 'levenshtein_street_stop_phonetic',
 'wratio_name',
 'jaro_winkler_name',
 'jaro_winkler_name_stop',
 'jaro_winkler_name_local',
 'jaro_winkler_name_local_transliterated',
 'jaro_winkler_name_reversed',
 'jw_name_local_ascii_only',
 'jw_name_local_nonascii_only',
 'tokenset_name_local',
 'tokenset_name_legal',
 'tokenset_name_local_nonascii_only',
 'tokenset_name_local_ascii_only',
 'tokenset_name_local_transliterated',
 'tokenset_name',
 'tokenset_name_stop',
 'tokenset_street_stop',
 'wratio_name_stop',
 'wratio_street_stop',
 'cosine_name_stop',
 'haversine',
 'same_phone']

In [89]:
features_df_w_semantic_sims.to_gbq("dh-global-sales-data-dev.leadgen_temp.AR_data_with_all_features_v5_e1_exp")

100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5849.80it/s]
